In [ ]:
# !pip install torch torchvision torchaudio
# !pip install transformers peft accelerate tqdm


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [ ]:
import json
import pickle
import os
from typing import Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from tqdm import tqdm

# ─────────────────────────────────────────────
#  CONFIG
# ─────────────────────────────────────────────

MODEL_NAME       = "Qwen/Qwen2.5-3B-Instruct"
TRAIN_FILE       = "train.jsonl"
TEST_FILE        = "test.jsonl"
OUTPUT_DIR       = "dual_lora_output"
PRED_OUTPUT_FILE = "preds.pkl"

LORA_R         = 8
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

EPOCHS         = 3      # total epochs
WARMUP_EPOCHS  = 1      # how many epochs to train lora_P on all tokens before specialising
BATCH_SIZE     = 4
GRAD_ACCUM     = 4
LR             = 2e-4
WARMUP_RATIO   = 0.05
MAX_SEQ_LEN    = 512
DTYPE          = torch.bfloat16

MAX_NEW_TOKENS = 256
DO_SAMPLE      = False

# ─────────────────────────────────────────────
#  DUAL-ADAPTER LINEAR LAYER
# ─────────────────────────────────────────────

class DualLoRALinear(nn.Module):
    """
    Frozen base Linear + two LoRA adapter pairs (P and C).

    _seq_len controls which adapter(s) are used:
      _seq_len == -2 -> warmup mode  -> lora_P only, on ALL tokens (no mask)
      _seq_len ==  0 -> prefill mode -> lora_P only (inference, prompt)
      _seq_len ==  1 -> decode mode  -> lora_C only (inference, generation)
      _seq_len >   1 -> train mode   -> blend via role_mask
      _seq_len == -1 -> not set      -> fallback lora_C (should not occur)
    """

    def __init__(self, base_linear: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base    = base_linear
        in_f         = base_linear.in_features
        out_f        = base_linear.out_features
        self.scaling = alpha / r

        dev  = next(base_linear.parameters()).device
        dtyp = next(base_linear.parameters()).dtype

        self.lora_P_A = nn.Linear(in_f, r,    bias=False, device=dev, dtype=dtyp)
        self.lora_P_B = nn.Linear(r,    out_f, bias=False, device=dev, dtype=dtyp)
        self.lora_C_A = nn.Linear(in_f, r,    bias=False, device=dev, dtype=dtyp)
        self.lora_C_B = nn.Linear(r,    out_f, bias=False, device=dev, dtype=dtyp)
        self.dropout  = nn.Dropout(dropout)

        nn.init.kaiming_uniform_(self.lora_P_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_P_B.weight)
        nn.init.kaiming_uniform_(self.lora_C_A.weight, a=5**0.5)
        nn.init.zeros_(self.lora_C_B.weight)

        for p in self.base.parameters():
            p.requires_grad = False

        self.role_mask: Optional[torch.Tensor] = None
        self._seq_len:  int = -1

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base_out = self.base(x)

        if self._seq_len == -2:
            # Warmup: lora_P only, applied to all tokens uniformly
            delta = self.lora_P_B(self.dropout(self.lora_P_A(x))) * self.scaling

        elif self._seq_len == 0:
            # Prefill: prompt tokens -> lora_P
            delta = self.lora_P_B(self.dropout(self.lora_P_A(x))) * self.scaling

        elif self._seq_len == 1:
            # Decode: one new token at a time -> lora_C
            delta = self.lora_C_B(self.dropout(self.lora_C_A(x))) * self.scaling

        elif self._seq_len > 1 and self.role_mask is not None and x.shape[1] == self._seq_len:
            # Specialisation training: blend by role_mask
            lora_P_out = self.lora_P_B(self.dropout(self.lora_P_A(x))) * self.scaling
            lora_C_out = self.lora_C_B(self.dropout(self.lora_C_A(x))) * self.scaling
            mask  = self.role_mask.unsqueeze(-1).to(x.dtype)   # (B, T, 1)
            delta = (1.0 - mask) * lora_P_out + mask * lora_C_out

        else:
            # Fallback - should not be reached in normal operation
            delta = self.lora_C_B(self.dropout(self.lora_C_A(x))) * self.scaling

        return base_out + delta


# ─────────────────────────────────────────────
#  DUAL-ADAPTER MODEL WRAPPER
# ─────────────────────────────────────────────

class DualLoRAModel(nn.Module):

    def __init__(self, base_model, target_modules, r, alpha, dropout):
        super().__init__()
        self.model = base_model
        self._dual_layers: list[DualLoRALinear] = []
        self._inject(target_modules, r, alpha, dropout)
        self._freeze_base()

    def _inject(self, target_modules, r, alpha, dropout):
        # Collect ALL (name, module) pairs before any setattr().
        # Mutating the module tree inside named_modules() iteration causes
        # layers to be visited multiple times, leaving _dual_layers with
        # stale references not in the live compute graph.
        replacements = []
        for name, module in self.model.named_modules():
            if not isinstance(module, nn.Linear):
                continue
            if not any(t in name for t in target_modules):
                continue
            replacements.append((name, module))

        for name, module in replacements:
            dual  = DualLoRALinear(module, r, alpha, dropout)
            self._dual_layers.append(dual)
            parts  = name.split(".")
            parent = self.model
            for part in parts[:-1]:
                parent = getattr(parent, part)
            setattr(parent, parts[-1], dual)

    def _freeze_base(self):
        for name, param in self.named_parameters():
            param.requires_grad = ("lora_P" in name or "lora_C" in name)

    def _set_mode(self, mode: str, role_mask: Optional[torch.Tensor] = None,
                  seq_len: int = -1):
        for layer in self._dual_layers:
            if mode == "warmup":
                # lora_P trains on all tokens; lora_C frozen
                layer.role_mask = None
                layer._seq_len  = -2
            elif mode == "train":
                layer.role_mask = role_mask
                layer._seq_len  = seq_len
            elif mode == "prefill":
                layer.role_mask = None
                layer._seq_len  = 0
            elif mode == "decode":
                layer.role_mask = None
                layer._seq_len  = 1

    def _clear(self):
        for layer in self._dual_layers:
            layer.role_mask = None
            layer._seq_len  = -1

    def _freeze_lora_C(self):
        """Freeze lora_C during warmup — only lora_P trains."""
        for name, param in self.named_parameters():
            if "lora_C" in name:
                param.requires_grad = False

    def _unfreeze_lora_C(self):
        """Unfreeze lora_C for specialisation phase."""
        for name, param in self.named_parameters():
            if "lora_C" in name:
                param.requires_grad = True

    def copy_P_to_C(self):
        """
        Copy lora_P weights into lora_C.
        Called once after warmup epochs complete, before specialisation begins.
        Both adapters now start from the same task-relevant initialisation.
        """
        for layer in self._dual_layers:
            layer.lora_C_A.weight.data.copy_(layer.lora_P_A.weight.data)
            layer.lora_C_B.weight.data.copy_(layer.lora_P_B.weight.data)
        print("Copied lora_P -> lora_C for warm-start specialisation.")

    # ── training forward ─────────────────────────────────────────────────────

    def forward(self, input_ids, attention_mask=None, labels=None,
                role_mask=None, mode="train", **kwargs):
        if mode == "warmup":
            self._set_mode("warmup")
        elif mode == "train" and role_mask is not None:
            self._set_mode("train", role_mask=role_mask,
                           seq_len=input_ids.shape[1])
        out = self.model(input_ids=input_ids, attention_mask=attention_mask,
                         labels=labels, **kwargs)
        self._clear()
        return out

    # ── two-phase inference ───────────────────────────────────────────────────

    def generate_dual(self, input_ids, attention_mask, max_new_tokens,
                      pad_token_id, eos_token_id):
        """
        Phase 1 - Prefill (lora_P):
            Run the full prompt through the model to populate the KV cache.
            No token is sampled here - this pass is purely for the cache.

        Phase 2 - Decode (lora_C):
            Switch adapter to lora_C. Sample token #1 from the prefill
            logits (last prompt position), then autoregressively generate
            all remaining tokens. Every generated token is under lora_C.
        """
        device = input_ids.device

        # Phase 1: prefill with lora_P
        self._set_mode("prefill")
        with torch.no_grad():
            prefill_out = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=True,
            )
        past_kv = prefill_out.past_key_values

        # Phase 2: decode with lora_C
        # Token #1 is sampled from the prefill logits (no re-feed of last
        # prompt token — the KV cache already contains it).
        self._set_mode("decode")
        next_token   = prefill_out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
        generated    = [next_token]
        current_attn = torch.cat(
            [attention_mask,
             torch.ones(input_ids.shape[0], 1, device=device, dtype=attention_mask.dtype)],
            dim=1
        )

        with torch.no_grad():
            for _ in range(max_new_tokens - 1):
                out = self.model(
                    input_ids=next_token,
                    attention_mask=current_attn,
                    past_key_values=past_kv,
                    use_cache=True,
                )
                past_kv    = out.past_key_values
                next_token = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                generated.append(next_token)
                current_attn = torch.cat(
                    [current_attn,
                     torch.ones(input_ids.shape[0], 1,
                                device=device, dtype=attention_mask.dtype)],
                    dim=1
                )
                if (next_token == eos_token_id).all():
                    break

        self._clear()
        return torch.cat(generated, dim=1)   # (B, gen_len)

    # ── persistence ──────────────────────────────────────────────────────────

    def save_adapters(self, path: str):
        os.makedirs(path, exist_ok=True)
        state = {k: v for k, v in self.state_dict().items()
                 if "lora_P" in k or "lora_C" in k}
        torch.save(state, os.path.join(path, "dual_lora_adapters.pt"))
        print(f"Saved {len(state)} adapter tensors -> {path}/dual_lora_adapters.pt")

    def load_adapters(self, path: str):
        state  = torch.load(os.path.join(path, "dual_lora_adapters.pt"),
                            map_location="cpu")
        missing, _ = self.load_state_dict(state, strict=False)
        lora_missing = [k for k in missing if "lora_P" in k or "lora_C" in k]
        print(f"Loaded {len(state)} adapter tensors. "
              f"LoRA keys missing (should be 0): {len(lora_missing)}")

    def trainable_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def total_parameters(self):
        return sum(p.numel() for p in self.parameters())


# ─────────────────────────────────────────────
#  DATASET
# ─────────────────────────────────────────────

class DualLoRADataset(Dataset):
    def __init__(self, filepath: str, tokenizer, max_len: int):
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.samples   = []
        with open(filepath) as f:
            for line in f:
                obj = json.loads(line.strip())
                self.samples.append((obj["prompt"], obj["completion"]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        prompt, completion = self.samples[idx]

        messages    = [{"role": "user", "content": prompt}]
        prompt_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

        prompt_ids = self.tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

        full_enc  = self.tokenizer(
            prompt_text + completion,
            max_length=self.max_len,
            truncation=True,
            add_special_tokens=False,
        )
        input_ids  = full_enc["input_ids"]
        attn_mask  = full_enc["attention_mask"]
        prompt_len = min(len(prompt_ids), len(input_ids))

        labels = [-100] * prompt_len + input_ids[prompt_len:]

        if input_ids[-1] != self.tokenizer.eos_token_id:
            if len(input_ids) < self.max_len:
                input_ids.append(self.tokenizer.eos_token_id)
                attn_mask.append(1)
                labels.append(self.tokenizer.eos_token_id)

        labels    = labels[:len(input_ids)]
        role_mask = [0] * prompt_len + [1] * (len(input_ids) - prompt_len)
        role_mask = role_mask[:len(input_ids)]

        return {
            "input_ids":      torch.tensor(input_ids,  dtype=torch.long),
            "attention_mask": torch.tensor(attn_mask,  dtype=torch.long),
            "labels":         torch.tensor(labels,     dtype=torch.long),
            "role_mask":      torch.tensor(role_mask,  dtype=torch.bool),
        }


def collate_fn(batch, pad_id):
    max_len   = max(b["input_ids"].shape[0] for b in batch)
    input_ids = torch.full((len(batch), max_len), pad_id, dtype=torch.long)
    attn_mask = torch.zeros(len(batch), max_len,          dtype=torch.long)
    labels    = torch.full((len(batch), max_len), -100,   dtype=torch.long)
    role_mask = torch.zeros(len(batch), max_len,          dtype=torch.bool)

    for i, b in enumerate(batch):
        n = b["input_ids"].shape[0]
        input_ids[i, :n] = b["input_ids"]
        attn_mask[i, :n] = b["attention_mask"]
        labels[i, :n]    = b["labels"]
        role_mask[i, :n] = b["role_mask"]

    return {"input_ids": input_ids, "attention_mask": attn_mask,
            "labels": labels, "role_mask": role_mask}


# ─────────────────────────────────────────────
#  TRAINING
# ─────────────────────────────────────────────

def train():
    print("Loading tokenizer and base model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
    )

    print("Injecting dual LoRA adapters...")
    model  = DualLoRAModel(base_model, TARGET_MODULES, LORA_R, LORA_ALPHA, LORA_DROPOUT)
    total  = model.total_parameters()
    train_ = model.trainable_parameters()
    print(f"Total: {total/1e6:.1f}M | Trainable: {train_/1e6:.1f}M ({100*train_/total:.2f}%)")
    print(f"Dual layers injected: {len(model._dual_layers)}")
    print(f"Warmup epochs: {WARMUP_EPOCHS} | Specialisation epochs: {EPOCHS - WARMUP_EPOCHS}")

    device  = next(model.parameters()).device
    dataset = DualLoRADataset(TRAIN_FILE, tokenizer, MAX_SEQ_LEN)
    loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                         collate_fn=lambda b: collate_fn(b, tokenizer.pad_token_id))

    # Total steps covers all epochs; scheduler runs continuously across both phases
    total_steps  = (len(loader) // GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    # ── Phase 1: warmup — train lora_P only on all tokens ────────────────────
    # lora_C is frozen during this phase
    model._freeze_lora_C()
    print(f"\nPhase 1: Warmup ({WARMUP_EPOCHS} epoch(s)) — lora_P trains on all tokens")

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
    )
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    model.train()
    for epoch in range(WARMUP_EPOCHS):
        total_loss = 0.0
        optimizer.zero_grad()
        pbar = tqdm(loader, desc=f"Warmup Epoch {epoch+1}/{WARMUP_EPOCHS}")

        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["labels"].to(device)
            # role_mask not used in warmup mode

            out  = model(input_ids=input_ids, attention_mask=attn_mask,
                         labels=labels, mode="warmup")
            loss = out.loss / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                pbar.set_postfix(loss=f"{total_loss/(step+1):.4f}",
                                 lr=f"{scheduler.get_last_lr()[0]:.2e}")

        print(f"Warmup Epoch {epoch+1} avg loss: {total_loss/len(loader):.4f}")

    # Copy lora_P -> lora_C, then unfreeze lora_C for specialisation
    model.copy_P_to_C()
    model._unfreeze_lora_C()

    # Rebuild optimizer to include lora_C parameters (now unfrozen)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01
    )
    # Continue scheduler from remaining steps
    remaining_steps = (len(loader) // GRAD_ACCUM) * (EPOCHS - WARMUP_EPOCHS)
    scheduler = get_cosine_schedule_with_warmup(optimizer, 0, remaining_steps)

    # ── Phase 2: specialisation — role-masked training ────────────────────────
    specialisation_epochs = EPOCHS - WARMUP_EPOCHS
    print(f"\nPhase 2: Specialisation ({specialisation_epochs} epoch(s)) — role-masked training")

    for epoch in range(specialisation_epochs):
        total_loss = 0.0
        optimizer.zero_grad()
        pbar = tqdm(loader, desc=f"Specialisation Epoch {epoch+1}/{specialisation_epochs}")

        for step, batch in enumerate(pbar):
            input_ids = batch["input_ids"].to(device)
            attn_mask = batch["attention_mask"].to(device)
            labels    = batch["labels"].to(device)
            role_mask = batch["role_mask"].to(device)

            out  = model(input_ids=input_ids, attention_mask=attn_mask,
                         labels=labels, role_mask=role_mask, mode="train")
            loss = out.loss / GRAD_ACCUM
            loss.backward()
            total_loss += loss.item() * GRAD_ACCUM

            if (step + 1) % GRAD_ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], 1.0
                )
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                pbar.set_postfix(loss=f"{total_loss/(step+1):.4f}",
                                 lr=f"{scheduler.get_last_lr()[0]:.2e}")

        print(f"Specialisation Epoch {epoch+1} avg loss: {total_loss/len(loader):.4f}")

    model.save_adapters(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("Training complete.")


# ─────────────────────────────────────────────
#  INFERENCE
# ─────────────────────────────────────────────

def infer():
    print("Loading tokenizer and base model...")
    tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True
    )

    model = DualLoRAModel(base_model, TARGET_MODULES, LORA_R, LORA_ALPHA, LORA_DROPOUT)
    model.load_adapters(OUTPUT_DIR)
    model.eval()

    device = next(model.parameters()).device

    with open(TEST_FILE) as f:
        lines = [json.loads(l.strip()) for l in f]

    results = []
    print(f"Generating on {len(lines)} test examples...")
    for i, obj in enumerate(tqdm(lines)):
        prompt      = obj["prompt"]
        messages    = [{"role": "user", "content": prompt}]
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        enc = tokenizer(prompt_text, return_tensors="pt",
                        add_special_tokens=False).to(device)

        gen_ids = model.generate_dual(
            input_ids      = enc["input_ids"],
            attention_mask = enc["attention_mask"],
            max_new_tokens = MAX_NEW_TOKENS,
            pad_token_id   = tokenizer.pad_token_id,
            eos_token_id   = tokenizer.eos_token_id,
        )

        prediction = tokenizer.decode(gen_ids[0], skip_special_tokens=True)

        if i < 5:
            print(prediction)
        results.append(prediction)

        if i >= 499:
            break

    with open(PRED_OUTPUT_FILE, "wb") as f:
        pickle.dump(results, f)
    print(f"Saved {len(results)} predictions -> {PRED_OUTPUT_FILE}")

In [3]:
train()

Loading tokenizer and base model...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Injecting dual LoRA adapters...
Total: 3115.9M | Trainable: 29.9M (0.96%)
Dual layers injected: 252
Warmup epochs: 2 | Specialisation epochs: 2

Phase 1: Warmup (2 epoch(s)) — lora_P trains on all tokens


Warmup Epoch 1/2: 100%|██████████| 250/250 [00:47<00:00,  5.26it/s, loss=0.7878, lr=1.79e-04]


...

Warmup Epoch 2 avg loss: 0.0956
Copied lora_P -> lora_C for warm-start specialisation.

Phase 2: Specialisation (2 epoch(s)) — role-masked training


Specialisation Epoch 1/2: 100%|██████████| 250/250 [01:15<00:00,  3.31it/s, loss=0.0723, lr=1.00e-04]


Specialisation Epoch 1 avg loss: 0.0723


Specialisation Epoch 2/2: 100%|██████████| 250/250 [01:12<00:00,  3.44it/s, loss=0.0326, lr=0.00e+00]


Specialisation Epoch 2 avg loss: 0.0325
Saved 1008 adapter tensors -> dual_lora_output/dual_lora_adapters.pt
Training complete.


In [ ]:
infer()

Loading tokenizer and base model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/tmp/ipykernel_2690/1956505744.py:305: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state  = torch.load(os.path.join(path, "dual_lora_adapters.pt"),


Loaded 1008 adapter tensors. LoRA keys missing (should be 0): 0
Generating on 416 test examples...


  0%|          | 1/416 [00:00<06:50,  1.01it/s]

...

  1%|          | 4/416 [00:03<06:26,  1.06it/s]

[{'event_type': 'Life:Die', 'trigger': 'killing'}]


  1%|          | 5/416 [00:04<06:19,  1.08it/s]

[{'event_type': 'Conflict:Attack', 'trigger': 'attacks'}]


100%|█████████▉| 415/416 [07:35<00:00,  1.05it/s]